In [ ]:
import os, time, math
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
from empath import Empath
from transformers import DistilBertTokenizer, DistilBertModel, get_linear_schedule_with_warmup

# --- CONFIGURATION ---
TEXT_COL = 'text'
LABEL_COL = 'real'
MAX_LEN = 80 # Reduced from 128 for speed
BATCH_SIZE = 32
EPOCHS = 5
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# --- 1. DATA LOADING & EMPATH FEATURES ---
df = pd.read_csv("data_cleaned.csv")
lexicon = Empath()

# Feature extraction: Empath Lexicon
REQUESTED_CATS = ["joy", "positive_emotion", "sadness", "anger", "negative_emotion", 
                  "social", "achievement", "affection", "violence", "disgust", "fear"]

def get_empath_features(texts):
    print("Extracting Empath features...")
    # Empath is CPU-bound; this is often the bottleneck before training starts
    features = []
    for text in texts:
        score = lexicon.analyze(str(text), categories=REQUESTED_CATS, normalize=True)
        if score is None: score = {c: 0.0 for c in REQUESTED_CATS}
        features.append([score.get(c, 0.0) for c in REQUESTED_CATS])
    return np.array(features)

E_features = get_empath_features(df[TEXT_COL])
scaler = StandardScaler()
E_scaled = scaler.fit_transform(E_features)

# --- 2. TRAIN/TEST SPLIT ---
X_train_text, X_test_text, E_train, E_test, y_train, y_test = train_test_split(
    df[TEXT_COL].values, E_scaled, df[LABEL_COL].values, test_size=0.2, random_state=42
)

# --- 3. PYTORCH DATASET ---
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

class FakeNewsDataset(Dataset):
    def __init__(self, texts, empath, labels):
        self.texts = texts
        self.empath = empath
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        encoding = tokenizer(text, truncation=True, padding='max_length', 
                             max_length=MAX_LEN, return_tensors='pt')
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'empath': torch.tensor(self.empath[idx], dtype=torch.float32),
            'label': torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_loader = DataLoader(FakeNewsDataset(X_train_text, E_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(FakeNewsDataset(X_test_text, E_test, y_test), batch_size=BATCH_SIZE)

# --- 4. OPTIMIZED HYBRID MODEL ---
class HybridModel(nn.Module):
    def __init__(self, num_classes, empath_dim):
        super(HybridModel, self). __init__()
        # DistilBert is much faster than standard BERT
        self.distilbert = DistilBertModel.from_pretrained("distilbert-base-uncased")
        
        # FREEZE BERT: This makes training 10x faster
        for param in self.distilbert.parameters():
            param.requires_grad = False
            
        self.lstm = nn.LSTM(768, 128, batch_first=True, bidirectional=True)
        self.gru = nn.GRU(256, 64, batch_first=True, bidirectional=True)
        
        self.empath_dense = nn.Linear(empath_dim, 32)
        self.classifier = nn.Sequential(
            nn.Linear(128 + 32, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )

    def forward(self, input_ids, attention_mask, empath):
        # Bert output
        bert_out = self.distilbert(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = bert_out.last_hidden_state # [batch, seq, 768]
        
        # Recurrent Layers
        lstm_out, _ = self.lstm(sequence_output)
        gru_out, _ = self.gru(lstm_out)
        avg_pool = torch.mean(gru_out, 1)
        
        # Empath side
        empath_out = torch.relu(self.empath_dense(empath))
        
        # Concatenate
        combined = torch.cat((avg_pool, empath_out), dim=1)
        return self.classifier(combined)

model = HybridModel(num_classes=2, empath_dim=len(REQUESTED_CATS)).to(DEVICE)

# --- 5. TRAINING WITH EARLY STOPPING ---
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4) # Slightly higher LR since BERT is frozen
criterion = nn.CrossEntropyLoss()

best_accuracy = 0
patience = 2
trigger_times = 0

print(f"Starting training on {DEVICE}...")
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(DEVICE)
        mask = batch['attention_mask'].to(DEVICE)
        empath = batch['empath'].to(DEVICE)
        labels = batch['label'].to(DEVICE)
        
        outputs = model(input_ids, mask, empath)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    # Validation
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for batch in test_loader:
            outputs = model(batch['input_ids'].to(DEVICE), batch['attention_mask'].to(DEVICE), batch['empath'].to(DEVICE))
            _, predicted = torch.max(outputs.data, 1)
            total += batch['label'].size(0)
            correct += (predicted == batch['label'].to(DEVICE)).sum().item()
    
    accuracy = correct / total
    print(f"Epoch {epoch+1} | Loss: {total_loss/len(train_loader):.4f} | Val Acc: {accuracy:.4f}")
    
    # Early Stopping Logic
    if accuracy > best_accuracy:
        best_accuracy = accuracy
        trigger_times = 0
        torch.save(model.state_dict(), 'best_hybrid_model.pt')
    else:
        trigger_times += 1
        if trigger_times >= patience:
            print("Early stopping!")
            break



        # SHAP introduction 
       # 1. Prepare data for SHAP
# Taking a small sample of 100 for the background and 50 for explanation to save time
batch = next(iter(train_loader))
background = [
    batch['input_ids'][:100].to(DEVICE),
    batch['attention_mask'][:100].to(DEVICE),
    batch['empath'][:100].to(DEVICE)
]

test_batch = next(iter(test_loader))
test_inputs = [
    test_batch['input_ids'][:50].to(DEVICE),
    test_batch['attention_mask'][:50].to(DEVICE),
    test_batch['empath'][:50].to(DEVICE)
]

# 2. Initialize Explainer
explainer = shap.DeepExplainer(model, background)

# 3. Calculate SHAP values
# This returns a list of arrays corresponding to [input_ids, attention_mask, empath]
shap_values = explainer.shap_values(test_inputs)

# 4. Extract Empath-specific importance (the 3rd input in your model)
# We take the mean absolute value across the test samples
empath_shap = np.abs(shap_values[2]).mean(axis=0)

# 5. Create a DataFrame and Save to CSV
shap_df = pd.DataFrame({
    'feature': REQUESTED_CATS,
    'mean_abs_shap_value': empath_shap
}).sort_values(by='mean_abs_shap_value', ascending=False)

shap_df.to_csv("empath_feature_importance.csv", index=False)
print("SHAP results saved to empath_feature_importance.csv")

# Optional: Quick visualization
shap_df.plot(kind='bar', x='feature', y='mean_abs_shap_value', title='Empath Feature Importance')
plt.show()

#Output
# 1. Run the model on the test batch
model.eval()
with torch.no_grad():
    test_ids = test_batch['input_ids'].to(DEVICE)
    test_mask = test_batch['attention_mask'].to(DEVICE)
    test_empath = test_batch['empath'].to(DEVICE)
    logits = model(test_ids, test_mask, test_empath)
    probs = torch.softmax(logits, dim=1).cpu().numpy()
    preds = torch.argmax(logits, dim=1).cpu().numpy()

# 2. Get the SHAP values for the Empath features

detailed_results = []

for i in range(len(preds)):
    # Get SHAP values for this specific article's predicted class
    article_empath_shap = shap_values[2][i]
   
    # Identify the top 3 influencing emotions (highest absolute SHAP value)
    top_indices = np.argsort(np.abs(article_empath_shap))[-3:][::-1]
    influences = [f"{REQUESTED_CATS[idx]} ({article_empath_shap[idx]:.4f})" for idx in top_indices]
   
    detailed_results.append({
        'Article_Index': i,
        'Original_Text_Snippet': X_test_text[i][:100] + "...", # First 100 chars
        'Prediction': "Real" if preds[i] == 1 else "Fake",
        'Confidence': f"{probs[i][preds[i]]*100:.2f}%",
        'Top_1_Influence': influences[0],
        'Top_2_Influence': influences[1],
        'Top_3_Influence': influences[2]
    })

# 3. Export to CSV
detailed_df = pd.DataFrame(detailed_results)
detailed_df.to_csv("article_classification_influence.csv", index=False)
print("Detailed breakdown saved to article_classification_influence.csv")

# Visualize the 1st article in the test batch
shap.initjs()
shap.force_plot(
    explainer.expected_value[preds[0]],
    shap_values[2][0],
    feature_names=REQUESTED_CATS
)

# --- DETAILED AUDIT WITH CONFIDENCE FILTER ---

THRESHOLD = 0.60  # Only include articles where the model is < 60% sure
unsure_results = []

model.eval()
with torch.no_grad():
    # Process the test batch
    logits = model(
        test_batch['input_ids'].to(DEVICE),
        test_batch['attention_mask'].to(DEVICE),
        test_batch['empath'].to(DEVICE)
    )
    probs = torch.softmax(logits, dim=1).cpu().numpy()
    preds = torch.argmax(logits, dim=1).cpu().numpy()

# shap_values[2] contains the contributions for the Empath input
for i in range(len(preds)):
    confidence = probs[i][preds[i]]
   
    # Apply the "Unsure" Filter
    if confidence < THRESHOLD:
        # Get SHAP values for this specific article's predicted class
        article_empath_shap = shap_values[2][i]
       
        # Identify the top 3 influencing emotions
        top_indices = np.argsort(np.abs(article_empath_shap))[-3:][::-1]
        influences = [f"{REQUESTED_CATS[idx]} ({article_empath_shap[idx]:.4f})" for idx in top_indices]
       
        unsure_results.append({
            'Article_Index': i,
            'Original_Text_Snippet': X_test_text[i][:150] + "...",
            'Prediction': "Real" if preds[i] == 1 else "Fake",
            'Confidence': f"{confidence*100:.2f}%",
            'Primary_Driver': influences[0],
            'Secondary_Driver': influences[1],
            'Tertiary_Driver': influences[2]
        })

# Save to a dedicated audit file
unsure_df = pd.DataFrame(unsure_results)
unsure_df.to_csv("unsure_predictions_audit.csv", index=False)

print(f"Found {len(unsure_results)} unsure predictions. Audit saved to unsure_predictions_audit.csv")



ModuleNotFoundError: No module named 'pandas'